# GFN2 fine-tune of MACE-OFF **off-large** on 250 rotaxane frames (Colab)

Same workflow as the off-medium notebook, scaled up to the deployed model size:

1. **GFN2-xTB labels** for `data/rot1_sampled_250.xyz` (torch-free subprocess)
2. **Reference pool** built from the MACE-OFF23 test split with *stock* `off-large` (3584-d latents)
3. **Baseline (before)**: per-frame molecule-mean OOD, all 250 frames
4. **Fine-tune** `off-large` on the GFN2 labels (energy **and** forces)
5. **After**: rebuild the pool *through the finetuned encoder* and re-score
6. **Size-matched stock control**: a CHNOF-only stock pool matched to the ft pool's atom count — the like-for-like baseline
7. **Per-atom OOD maps** (pairs for frames 0-9) + histogram
8. **Save everything by direct download** (browser): checkpoint, scores, images

Everything is computed up here; you only download the results (small: checkpoint ~50 MB,
scores, image zip — the big pools stay on the VM). Direct download goes to your browser's
download folder — no Drive involved.

**First: Runtime → Change runtime type → GPU (A100).**

In [ ]:
# @title 1. Clone the repo + install the stack {display-mode:"form"}
import os, subprocess, torch

print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("WARNING: no GPU attached — the fine-tune cell will be ~10x slower.")

if not os.path.exists("/content/mace/code/mace_calc.py"):
    subprocess.run(["git", "clone", "-q",
                    "https://github.com/MauricioCafiero/MACE_UseAndTrain.git",
                    "/content/mace"], check=True)
    print("repo cloned")
%cd /content/mace
!pip -q install mace-torch tblite 2>&1 | tail -3
print("install done")

### Carried-in lessons (medium run + probe)

- **Forces must carry real weight** (`forces_weight=100`): the energy-only probe wrecked
  the forces (RMSE 175 → 3840 meV/Å) and shifted the whole latent manifold, sending OOD *up*.
- **Pool matching rule**: any before/after comparison must match the pool in encoder, size
  AND composition. The ft checkpoint's z_table only knows H,C,N,O/F (`--E0s=average`), so its
  pool drops the P/S/Cl/Br/I OFF23 frames — expect "frame k failed (16 is not in list)"
  during the ft pool build, and a size-matched stock control to compensate.
- off-medium went in "A100 minutes"; off-large is bigger (~3x params) but 30 epochs over
  225 frames should still be well under an hour.

In [ ]:
# @title 2. GFN2-xTB labels for the 250 frames (~3 min) {display-mode:"form"}
# tblite and torch bundle separate OpenMP runtimes and segfault when loaded
# into one process, so labeling runs as its own torch-free worker pool.
!mkdir -p runs
!python code/gfn2_label.py data/rot1_sampled_250.xyz runs/rot250_gfn2.xyz --workers 4

In [ ]:
# @title 3. Split into train / valid (every 10th frame held out) {display-mode:"form"}
import sys; sys.path.insert(0, "code")
from ase.io import read, write

frames = read("runs/rot250_gfn2.xyz", index=":")
valid = [a for i, a in enumerate(frames) if i % 10 == 0]
train = [a for i, a in enumerate(frames) if i % 10 != 0]
write("runs/rot250_gfn2_train.xyz", train, format="extxyz")
write("runs/rot250_gfn2_valid.xyz", valid, format="extxyz")
print(f"{len(train)} train / {len(valid)} valid frames")

In [ ]:
# @title 4. Build the stock off-large reference pool (one-time, ~15-30 min) {display-mode:"form"}
# Downloads the MACE-OFF23 test split (~81 MB) and encodes 2500 drug-like
# frames with the stock off-large encoder (3584-d per-atom latents).
import sys; sys.path.insert(0, "code")
from activation_ood import ReferencePool

pool = ReferencePool.build(n_frames=2500, model="off-large",
                           out="data/off23_pool_large.npz")
print("pool atoms:", pool.atom_vecs.shape[0])

In [ ]:
# @title 5. Baseline: score all 250 frames with stock off-large ("before", ~3 min) {display-mode:"form"}
import numpy as np
import mace_calc as mc
from trust_frames import read_frames
from activation_ood import atom_ood_scores

frames = read_frames("data/rot1_sampled_250.xyz")
mc.attach(frames[0], model="off-large", device="cuda", dtype="float32")
shared = frames[0].calc            # one calculator shared across all frames

before_big = []
for k, at in enumerate(frames):
    at.calc = shared
    d = atom_ood_scores(at, pool)["distances"]
    before_big.append(float(np.nanmean(d)))
    if k % 50 == 0:
        print(f"frame {k:3d}  mean OOD {before_big[-1]:.3f}")
before_big = np.array(before_big)
print(f"\nSTOCK off-large / big pool: mean {before_big.mean():.3f}  "
      f"range [{before_big.min():.3f}, {before_big.max():.3f}]")

In [ ]:
# @title 6. Fine-tune off-large on the GFN2 labels (GPU) {display-mode:"form"}
# batch_size 4: off-large is ~3x off-medium; keep memory headroom on the 144-atom frames.
from finetune_mace import run_finetune, find_latest_model

run_finetune(
    "runs/rot250_gfn2_train.xyz", "runs/rot250_gfn2_valid.xyz",
    foundation_model="off-large", name="rot250L", results_dir="runs",
    max_num_epochs=30,
    energy_weight=100.0, forces_weight=100.0,
    scheduler="ReduceLROnPlateau",
    device="cuda", default_dtype="float32",
    extra=("--batch_size=4", "--valid_batch_size=4", "--eval_interval=2"),
)
model_path = find_latest_model("runs", "rot250L")
print("finetuned checkpoint:", model_path)

In [ ]:
# @title 7. Rebuild the pool through the finetuned encoder + re-score ("after") {display-mode:"form"}
# "frame k failed (16 is not in list)" warnings are EXPECTED: the ft checkpoint's
# atomic-energies table only knows its training elements (H,C,N,O,F), so OFF23
# frames with P/S/Cl/Br/I are skipped and the ft pool is CHNOF-only + smaller.
# Cell 7b builds the size-matched stock control for the like-for-like number.
from pathlib import Path
from mace.calculators import MACECalculator
import mace_calc as mc
import matplotlib.pyplot as plt

ft = MACECalculator(model_paths=str(model_path), device="cuda",
                    default_dtype="float32")
stock_get_calculator = mc.get_calculator   # restore before the 7b control
mc.get_calculator = lambda **kw: ft        # pool build runs the finetuned encoder

ft_pool = ReferencePool.build(n_frames=2500, model="off-large",
                              out="data/off23_pool_ftL.npz")
print("ft pool atoms:", ft_pool.atom_vecs.shape[0])

mc.get_calculator = stock_get_calculator
after = []
for k, at in enumerate(frames):
    at.calc = ft
    after.append(float(np.nanmean(atom_ood_scores(at, ft_pool)["distances"])))
after = np.array(after)
print(f"\nFINETUNED off-large / own pool: mean {after.mean():.3f}  "
      f"range [{after.min():.3f}, {after.max():.3f}]")

plt.figure(figsize=(7.5, 4))
plt.hist(before_big, bins=30, alpha=0.55,
         label=f"stock off-large / big pool ({before_big.mean():.3f})")
if "before_matched" in globals():
    plt.hist(before_matched, bins=30, alpha=0.55,
             label=f"stock / size-matched CHNOF pool ({before_matched.mean():.3f})")
plt.hist(after, bins=30, alpha=0.55,
         label=f"finetuned / own pool ({after.mean():.3f})")
plt.axvline(0.25, ls="--", c="k", lw=1)
plt.text(0.255, plt.ylim()[1] * 0.9, "trust line (0.25)", fontsize=8)
plt.xlabel("molecule-mean latent OOD (cosine)")
plt.ylabel("frames")
plt.title("250 rotaxane frames, off-large: before vs after GFN2 fine-tune")
plt.legend(fontsize=8)
plt.show()

In [ ]:
# @title 7b. Size-matched STOCK off-large CHNOF pool — the like-for-like baseline {display-mode:"form"}
# The ft pool is CHNOF-only (z_table) and smaller than the big stock pool, so
# score the stock model against a stock-built CHNOF pool with (approximately)
# the same atom count. Frame count estimated from the ratio of pool sizes.
import sys; sys.path.insert(0, "code")
import numpy as np
import mace_calc as mc
from pathlib import Path
import activation_ood as ao
from activation_ood import ReferencePool

mc.get_calculator = stock_get_calculator      # stock encoder for this pool

# CHNOF-only filter, same as the local matched control
_orig_iter = ao._iter_frames
def _chnof_only(*a, **kw):
    for fr in _orig_iter(*a, **kw):
        if set(fr.get_atomic_numbers()) <= {1, 6, 7, 8, 9}:
            yield fr
ao._iter_frames = _chnof_only

big = pool.atom_vecs.shape[0]
n_est = max(1, round(2500 * ft_pool.atom_vecs.shape[0] / big))
print(f"target ~{ft_pool.atom_vecs.shape[0]} atoms -> trying n_frames={n_est}")
before_pool = ReferencePool.build(n_frames=n_est, model="off-large",
                                  out="data/off23_pool_large_CHNOF_matched.npz")
print("matched pool atoms:", before_pool.atom_vecs.shape[0],
      "| ft pool:", ft_pool.atom_vecs.shape[0])
ao._iter_frames = _orig_iter

mc.attach(frames[0], model="off-large", device="cuda", dtype="float32")
shared = frames[0].calc
before_matched = []
for k, at in enumerate(frames):
    at.calc = shared
    d = atom_ood_scores(at, before_pool)["distances"]
    before_matched.append(float(np.nanmean(d)))
    if k % 50 == 0:
        print(f"frame {k:3d}  mean OOD {before_matched[-1]:.3f}")
before_matched = np.array(before_matched)
print(f"\nSTOCK off-large / SIZE-MATCHED CHNOF pool: mean {before_matched.mean():.3f}")
print(f"\nLIKE-FOR-LIKE: after {after.mean():.3f} - before_matched "
      f"{before_matched.mean():.3f} = {after.mean() - before_matched.mean():+.3f}")

In [ ]:
# @title 8. Per-atom OOD map pairs, frames 0-9 (before / after) {display-mode:"form"}
# IMPORTANT: render the BEFORE images FIRST, while mc.get_calculator still
# returns the stock off-large calculator; only then point everything at the
# finetuned model for the AFTER images. (ood_map builds its calculator via
# mc.get_calculator and reads args from sys.argv.)
from pathlib import Path
import sys, shutil
import trust
import ood_map

Path("viz_pairs").mkdir(exist_ok=True)

# ---- before: stock off-large + stock big pool (no monkeypatch active) ----
for frame in range(10):
    tmp = Path(f"/tmp/ood_big_{frame}")
    shutil.rmtree(tmp, ignore_errors=True)   # fresh dir => no stale scoring cache
    sys.argv = ["ood_map", "data/rot1_sampled_250.xyz", "--model", "off-large",
                "--frame", str(frame), "--out-dir", str(tmp)]
    ood_map.main()
    shutil.move(tmp / "frame00_ood.png", f"viz_pairs/frame{frame:02d}_before.png")
    print("--- before frame", frame, "done", flush=True)

# ---- after: finetuned checkpoint + its own pool ----
trust.POOLS["rot250L-ft"] = Path("data/off23_pool_ftL.npz")
mc.get_calculator = lambda **kw: ft          # ood_map's mc is this same module

for frame in range(10):
    tmp = Path(f"/tmp/ood_ft_{frame}")
    shutil.rmtree(tmp, ignore_errors=True)
    sys.argv = ["ood_map", "data/rot1_sampled_250.xyz", "--model", "rot250L-ft",
                "--frame", str(frame), "--out-dir", str(tmp)]
    ood_map.main()
    shutil.move(tmp / "frame00_ood.png", f"viz_pairs/frame{frame:02d}_after.png")
    print("--- after frame", frame, "done", flush=True)

print(sorted(p.name for p in Path("viz_pairs").iterdir()))

In [ ]:
# @title 9. Download the checkpoint + scores + images {display-mode:"form"}
# Direct browser download (no Google Drive involved). The browser's "multiple
# downloads" permission prompt may appear -- accept it. If any file is missed
# (browser blocks popups), just rerun this cell; the files are recomputed in
# place, not re-trained.
from google.colab import files
import numpy as np, shutil

np.savez("rot250L_ood_scores.npz", before_big=before_big, after=after,
         **({"before_matched": before_matched} if "before_matched" in globals() else {}))
shutil.make_archive("rot250L_images", "zip", ".", "viz_pairs")

files.download(str(model_path))
files.download("rot250L_ood_scores.npz")
files.download("rot250L_images.zip")

### Reading the result (off-large)

- **Like-for-like = `after` vs `before_matched`** (cell 7b): both scored against
  CHNOF pools of the same size, each through its own encoder. The raw
  `after` vs big-pool `before_big` comparison mixes in the pool-size artifact
  (kNN distances inflate as pools shrink).
- Expected baseline: stock off-large scores slightly *lower* than off-medium's
  0.148 (tighter latents; S66 MAE 0.22 vs 0.26). The number that matters is the
  matched delta, and whether it reproduces off-medium's **−0.028** (and the
  dethreading held-out set's −0.030).
- Same caveats: scored frames are the training frames; the ft model speaks
  C/H/N/O/F only. If the matched delta holds, the fine-tune effect is
  model-size-independent.